In [1]:
# Imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from functools import partial
from seed import seed_database
from tools import execute_tool, TOOLS
from assistant import chat_with_chef, add_user_message, generate_response
from images import generate_recipe_image
from audio import transcribe_recording, generate_speech

In [2]:
# Constants
## Models 
AI_CHEF_MODEL="gpt-6-luna"
AI_CHEF_IMAGE_MODEL="gpt-image-1-mini"
AI_CHEF_STT_MODEL="gpt-4o-mini-transcribe"
AI_CHEF_TTS_MODEL="gpt-4o-mini-tts"

In [ ]:
# AI Chef Assistant
## 1. Database Init

seed_database()

## 2. Initialization of the AI

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

client = OpenAI()

In [4]:
## system message
SYSTEM_PROMPT = """
You are AI Chef, a friendly and knowledgeable cooking assistant.

Your responsibilities:
- Suggest recipes based on available ingredients.
- Help users prepare healthy and delicious meals.
- Adapt recipes to calorie and macronutrient targets.
- Provide clear, step-by-step cooking instructions.
- Use available tools to search for ingredients and calculate nutrition.

Important rules:
- Never invent nutritional values.
- Use the food database tools for nutritional calculations.
- Clearly identify ingredients missing from the database.
- Distinguish raw and cooked ingredient quantities.
- Keep recipes practical and easy to follow.
- Respond in the same language as the user.
- Prefer ingredients already available in the food database.
- When the user specifies ingredients, prioritize those ingredients.
- Do not search for unrelated alternatives unless requested.
- If an ingredient is missing, report it rather than repeatedly searching.

Recipe output rules:
- When providing a complete recipe with ingredients and cooking
  instructions, start your response with [RECIPE].
- For ordinary questions, food searches, or nutritional facts,
  do not include this marker.
- Only include [RECIPE] for complete recipes.
- Never explain the marker to the user.
"""

In [ ]:
## Simple test of the LLM
response = client.chat.completions.create(
    model=AI_CHEF_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Suggest a simple high-protein dinner using chicken, rice and tomatoes. Keep it brief."
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
## Testing tools
print("Registered tools:", [
    tool["function"]["name"] for tool in TOOLS
])

print(execute_tool("search_food", {"query": "chicken"}))

In [ ]:
## Test chat with the chef model function
answer, history = chat_with_chef(
    "Check your database for chicken breast and tell me its nutritional values per 100g.",
    client,
    AI_CHEF_MODEL,
    SYSTEM_PROMPT
)

print("\nAI Chef:")
print(answer)


In [5]:
# Configure the Gradio callbacks
handle_response = partial(
    generate_response,
    client=client,
    model=AI_CHEF_MODEL,
    system_prompt=SYSTEM_PROMPT
)

handle_image = partial(
    generate_recipe_image,
    client=client,
    model=AI_CHEF_IMAGE_MODEL
)

handle_transcription = partial(
    transcribe_recording,
    client=client,
    model=AI_CHEF_STT_MODEL
)

handle_speech = partial(
    generate_speech,
    client=client,
    model=AI_CHEF_TTS_MODEL
)


def generate_speech_with_button(recipe):
    audio_path = handle_speech(recipe)

    return (
        audio_path,
        gr.update(interactive=bool(audio_path))
    )


In [6]:
# custom css
CUSTOM_CSS = """
    #meal-panel {
        position: relative;
    }

    #meal-audio {
        position: absolute;
        top: 12px;
        right: 12px;
        width: 200px;
        z-index: 10;
    }

    #record-btn, #send-btn {
        width: 46px !important;
        min-width: 46px !important;
        height: 46px !important;
        padding: 0 !important;
        border-radius: 12px !important;
        display: flex;
        align-items: center;
        justify-content: center;
    }

    #record-btn::before,
    #send-btn::before {
        content: "";
        display: block;
        width: 20px;
        height: 20px;
        background-color: currentColor;
        mask-repeat: no-repeat;
        mask-position: center;
        mask-size: contain;
    }

    #record-btn::before {
        mask-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24'%3E%3Crect x='9' y='2' width='6' height='12' rx='3' fill='black'/%3E%3Cpath d='M5 10a7 7 0 0 0 14 0M12 17v5M8 22h8' stroke='black' stroke-width='2' fill='none'/%3E%3C/svg%3E");
    }

    #send-btn::before {
        mask-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24'%3E%3Cpath d='M22 2L11 13M22 2L15 22L11 13L2 9L22 2Z' stroke='black' stroke-width='2' fill='none'/%3E%3C/svg%3E");
    }

    #record-btn.recording::before {
        mask-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24'%3E%3Crect x='5' y='5' width='14' height='14' rx='2' fill='black'/%3E%3C/svg%3E");
    }

    #record-btn.recording {
        background: #b91c1c !important;
        color: white !important;
    }

    /* Hide native audio controls, but keep audio available */
        #speech-audio {
        position: fixed !important;
        left: -10000px !important;
        top: 0 !important;
        width: 1px !important;
        height: 1px !important;
        overflow: hidden !important;
        opacity: 0 !important;
        pointer-events: none !important;
    }

    /* Floating play/pause button */
    #play-btn {
        position: absolute !important;
        bottom: 16px;
        right: 16px;
        z-index: 20;
        width: 48px !important;
        min-width: 48px !important;
        height: 48px !important;
        padding: 0 !important;
        border-radius: 12px !important;
        background: #ff6500 !important;
        color: white !important;
        display: flex;
        align-items: center;
        justify-content: center;
    }

    #play-btn::before {
        content: "";
        width: 22px;
        height: 22px;
        display: block;
        background: currentColor;
        mask-repeat: no-repeat;
        mask-position: center;
        mask-size: contain;

        /* Play icon */
        mask-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24'%3E%3Cpath fill='black' d='M8 5v14l11-7z'/%3E%3C/svg%3E");
    }

    /* Pause icon */
    #play-btn.playing::before {
        mask-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24'%3E%3Cpath fill='black' d='M6 4h4v16H6zM14 4h4v16h-4z'/%3E%3C/svg%3E");
    }

    #play-btn:disabled {
        opacity: 0.4 !important;
        cursor: not-allowed !important;
    }
"""

In [7]:
# custom js for the buttons
RECORD_JS = """
async () => {
    const button = document.querySelector("#record-btn");

    if (window.chefRecorder?.state === "recording") {
        const recorder = window.chefRecorder;

        const audio = await new Promise(resolve => {
            recorder.onstop = () => {
                const blob = new Blob(window.chefChunks, {
                    type: recorder.mimeType
                });

                const reader = new FileReader();
                reader.onloadend = () => resolve(reader.result);
                reader.readAsDataURL(blob);
            };

            recorder.stop();
        });

        window.chefStream.getTracks().forEach(t => t.stop());
        window.chefRecorder = null;

        button.classList.remove("recording");

        return audio;
    }

    try {
        const stream = await navigator.mediaDevices.getUserMedia({
            audio: true
        });

        window.chefStream = stream;
        window.chefChunks = [];

        const recorder = new MediaRecorder(stream);
        window.chefRecorder = recorder;

        recorder.ondataavailable = event => {
            if (event.data.size > 0) {
                window.chefChunks.push(event.data);
            }
        };

        recorder.start();

        button.classList.add("recording");

        return "";
    } catch (error) {
        console.error("Microphone error:", error);
        button.classList.remove("recording");
        return "";
    }
}
"""

# play button js
PLAY_JS = """
() => {
    const button = document.querySelector("#play-btn");
    const player = document.querySelector("#speech-audio");

    if (!button || !player) return;

    // Get the generated MP3 URL from Gradio
    const downloadLink = player.querySelector("a.download-link");
    if (!downloadLink) return;

    // Use our own audio element for playback
    if (!window.chefAudio) {
        window.chefAudio = new Audio();
    }

    const audio = window.chefAudio;
    const audioUrl = downloadLink.href;

    // Load the latest generated audio
    if (audio.src !== audioUrl) {
        audio.pause();
        audio.src = audioUrl;
        audio.load();
        button.classList.remove("playing");
    }

    // Update icon when playback ends
    audio.onended = () => {
        button.classList.remove("playing");
    };

    if (audio.paused) {
        audio.play()
            .then(() => button.classList.add("playing"))
            .catch(console.error);
    } else {
        audio.pause();
        button.classList.remove("playing");
    }
}
"""


In [ ]:
## Gradio markup and events
with gr.Blocks(
    title="AI Chef Assistant",
    css=CUSTOM_CSS
) as demo:

    gr.Markdown("# AI Chef Assistant")
    gr.Markdown(
        "Tell me what ingredients you have, "
        "and I'll help you create a meal."
    )

    conversation_state = gr.State([])
    recipe_state = gr.State(None)
    recorded_audio = gr.Textbox(visible=False)

    with gr.Row(equal_height=True):

        # LEFT — CHAT (2/3)
        with gr.Column(scale=2, min_width=450):

            chatbot = gr.Chatbot(
                label="AI Chef",
                type="messages",
                height=500
            )

            with gr.Row(equal_height=True):
                message_input = gr.Textbox(
                    placeholder="Your message...",
                    show_label=False,
                    container=False,
                    scale=8,
                    min_width=250
                )

                record_button = gr.Button(
                    "",
                    variant="secondary",
                    elem_id="record-btn",
                    min_width=48,
                    scale=0
                )

                send_button = gr.Button(
                    "",
                    variant="primary",
                    elem_id="send-btn",
                    min_width=48,
                    scale=0
                )

        # RIGHT — IMAGE + PLAYBACK (1/3)
        with gr.Column(
            scale=1,
            min_width=260,
            elem_id="meal-panel"
        ):

            meal_image = gr.Image(
                label="Generated Meal",
                interactive=False,
                height=500
            )

            speech_audio = gr.Audio(
                interactive=False,
                autoplay=False,
                show_label=False,
                elem_id="speech-audio"
            )

            play_button = gr.Button(
                "",
                variant="primary",
                elem_id="play-btn",
                min_width=48,
                scale=0,
                interactive=False
            )
    
    # Send button
    send_event = send_button.click(
        fn=lambda message, history: (
            *add_user_message(message, history),
            None,
            None,
            gr.update(interactive=False)
        ),
        inputs=[message_input, chatbot],
        outputs=[message_input, chatbot, recipe_state, meal_image, play_button],
        queue=False
    )

    response_event = send_event.then(
        fn=handle_response,
        inputs=[chatbot, conversation_state],
        outputs=[chatbot, conversation_state, recipe_state]
    )

    image_event = response_event.then(
        fn=handle_image,
        inputs=[recipe_state],
        outputs=[meal_image]
    )

    
    speech_event = response_event.then(
        fn=generate_speech_with_button,
        inputs=[recipe_state],
        outputs=[speech_audio, play_button]
    )

    # Enter key
    submit_event = message_input.submit(
        fn=lambda message, history: (
            *add_user_message(message, history),
            None,
            None,
            gr.update(interactive=False)
        ),
        inputs=[message_input, chatbot],
        outputs=[message_input, chatbot, recipe_state, meal_image, play_button],
        queue=False
    )

    submit_response_event = submit_event.then(
        fn=handle_response,
        inputs=[chatbot, conversation_state],
        outputs=[chatbot, conversation_state, recipe_state]
    )

    image_event = submit_response_event.then(
        fn=handle_image,
        inputs=[recipe_state],
        outputs=[meal_image]
    )

    
    speech_event = submit_response_event.then(
        fn=generate_speech_with_button,
        inputs=[recipe_state],
        outputs=[speech_audio, play_button]
    )

    # Record button — start/stop recording
    record_button.click(
        fn=None,
        inputs=[],
        outputs=[recorded_audio],
        js=RECORD_JS,
        queue=False
    )

    # When recording stops, transcribe audio
    transcription_event = recorded_audio.change(
        fn=handle_transcription,
        inputs=[recorded_audio],
        outputs=[message_input]
    )

    # Display the transcribed message
    voice_event = transcription_event.then(
        fn=add_user_message,
        inputs=[message_input, chatbot],
        outputs=[message_input, chatbot],
        queue=False
    )

    # Hide Play button after voice submission
    voice_event.then(
        fn=lambda: gr.update(interactive=False),
        outputs=[play_button],
        queue=False
    )

    # Process with AI Chef
    voice_response_event = voice_event.then(
        fn=handle_response,
        inputs=[chatbot, conversation_state],
        outputs=[chatbot, conversation_state, recipe_state]
    )

    # Generate image
    voice_image_event = voice_response_event.then(
        fn=handle_image,
        inputs=[recipe_state],
        outputs=[meal_image]
    )

    voice_speech_event = voice_response_event.then(
        fn=generate_speech_with_button,
        inputs=[recipe_state],
        outputs=[speech_audio, play_button]
    )

        # Play/pause recipe narration
    play_button.click(
        fn=None,
        inputs=[],
        outputs=[],
        js=PLAY_JS,
        queue=False
    )
    

demo.launch(inbrowser=True)